# Feature Extraction

In this notebook we will extract the MFCC, LFCC, and CQCC features from the preprocessed data.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path("/scratch1/kodachi/p2")
MANIFESTS_DIR = PROJECT_ROOT / "manifests"
WAV16_DIR = PROJECT_ROOT / "data" / "wav16"
FEATURES_DIR = PROJECT_ROOT / "features"
FEATURES_DIR.mkdir(exist_ok=True, parents=True)

# Which preprocessed manifests to run on:
TRAIN_PRE = MANIFESTS_DIR / "LA_train_preprocessed.csv"
DEV_PRE   = MANIFESTS_DIR / "LA_dev_preprocessed.csv"
EVAL_PRE  = MANIFESTS_DIR / "LA_eval_preprocessed.csv"

# Frame params (standard for speech)
SR = 16000
N_FFT = 512           # window for STFT (~32ms); keep consistent with literature choices
HOP_LENGTH = 160      # 10 ms hop @16k
WIN_LENGTH = 400      # 25 ms window @16k

# Cepstral dims
MFCC_N_MELS = 40
MFCC_N_CEP = 20

LFCC_N_FILTERS = 40   # number of linear filterbanks
LFCC_N_CEP = 20

CQCC_N_CEP = 20       # how many cepstral coefficients to keep after DCT on log-CQT

# Pooling function (mean + std)
def pool_mean_std(feat: np.ndarray):
    # feat: (T, D) => return (2*D,)
    mean = np.mean(feat, axis=0)
    std  = np.std(feat, axis=0)
    return np.concatenate([mean, std], axis=0)

print("Setup complete. SR=", SR, "hop=", HOP_LENGTH, "win=", WIN_LENGTH)

Setup complete. SR= 16000 hop= 160 win= 400


## MFCC

In [ ]:
import librosa
import os
import json

def extract_mfcc_from_wav(wav_path, sr=SR, n_mfcc=MFCC_N_CEP, n_mels=MFCC_N_MELS,
                          n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH):
    y, sr0 = librosa.load(wav_path, sr=sr, mono=True)
    # compute mfcc: shape (n_mfcc, T)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_mels=n_mels,
                                n_fft=n_fft, hop_length=hop_length, win_length=win_length)
    # transpose to (T, D)
    return mfcc.T.astype(np.float32)

def run_mfcc_on_manifest(manifest_csv, out_dir: Path, pooled=True, save_frames=False, att_type = None):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(manifest_csv)
    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"MFCC {manifest_csv.name}"):
        utt = r['utt']
        wav = r['wav']
        if att_type:
            attack = r["attack_type"]
        label = int(r['label'])
        try:
            frames = extract_mfcc_from_wav(wav)
            if save_frames:
                np.save(out_dir / f"{utt}_frames.npy", frames)
            vec = pool_mean_std(frames) if pooled else frames
            np.save(out_dir / f"{utt}.npy", vec)
            if att_type:
                rows.append((utt, str((out_dir / f'{utt}.npy').resolve()), label, attack))
            else:
                rows.append((utt, str((out_dir / f'{utt}.npy').resolve()), label))
        except Exception as e:
            print(f"Failed {utt}: {e}")
    # write a small manifest for these features
    out_manifest = out_dir.parent / f"{out_dir.name}_{manifest_csv.stem}_feat_manifest.csv"
    if att_type:
        pd.DataFrame(rows, columns=["utt","feat_path","label", "attack_type"]).to_csv(out_manifest, index=False)
    else:
        pd.DataFrame(rows, columns=["utt","feat_path","label"]).to_csv(out_manifest, index=False)
    print("Wrote MFCC features to", out_dir, "and manifest", out_manifest)
    return out_manifest

In [ ]:
# mfcc_train_manifest = run_mfcc_on_manifest(TRAIN_PRE, FEATURES_DIR / "mfcc", pooled=True, save_frames=False)

In [ ]:
# mfcc_dev_manifest = run_mfcc_on_manifest(DEV_PRE, FEATURES_DIR / "mfcc", pooled=True, save_frames=False)

In [ ]:
# mfcc_dev_manifest = run_mfcc_on_manifest(EVAL_PRE, FEATURES_DIR / "mfcc", pooled=True, save_frames=False,att_type=True)

## LFCC

In [ ]:
import scipy.fftpack as fftpack

def linear_filterbank(n_filters=LFCC_N_FILTERS, n_fft=N_FFT, sr=SR):
    # Build center frequencies linearly spaced from 0..sr/2
    # Create triangular filters across FFT bins.
    freqs = np.linspace(0, sr/2, int(n_fft//2 + 1))
    # choose filter center frequencies equally spaced
    centers = np.linspace(0, sr/2, n_filters + 2)  # include edges
    fb = np.zeros((n_filters, len(freqs)), dtype=np.float32)
    for i in range(n_filters):
        left = centers[i]
        center = centers[i+1]
        right = centers[i+2]
        # triangle
        up_slope = (freqs - left) / (center - left + 1e-9)
        down_slope = (right - freqs) / (right - center + 1e-9)
        tri = np.maximum(0, np.minimum(up_slope, down_slope))
        fb[i, :] = tri
    return fb, freqs

def extract_lfcc_from_wav(wav_path, sr=SR, n_filters=LFCC_N_FILTERS, n_cep=LFCC_N_CEP,
                          n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH):
    y, sr0 = librosa.load(wav_path, sr=sr, mono=True)
    # compute power spectrogram (shape: (freq_bins, T))
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length, win_length=win_length, center=True))**2
    fb, freqs = linear_filterbank(n_filters=n_filters, n_fft=n_fft, sr=sr)
    # map S bins to fb via interpolation: S has freq bins corresponding to FFT bins
    # fft_freqs:
    fft_freqs = np.linspace(0, sr/2, S.shape[0])
    # interpolate filterbank to match S.shape[0] if needed (our fb already matches)
    # apply filterbank: filterbank @ S  -> (n_filters, T)
    S_power = S[:fb.shape[1], :] if S.shape[0] >= fb.shape[1] else S
    # If shapes mismatch, resample fb to S.shape[0]
    if fb.shape[1] != S.shape[0]:
        # interpolate fb over fft_freqs
        fb_interp = np.zeros((fb.shape[0], S.shape[0]), dtype=np.float32)
        fb_freqs = freqs
        target_freqs = fft_freqs
        for i in range(fb.shape[0]):
            fb_interp[i, :] = np.interp(target_freqs, fb_freqs, fb[i, :])
        fb_use = fb_interp
    else:
        fb_use = fb
    filt_energies = np.dot(fb_use, S)
    # avoid zeros before log
    filt_energies[filt_energies <= 1e-12] = 1e-12
    logE = np.log(filt_energies)
    # DCT on log energies along filter axis -> cepstral coefficients
    cep = fftpack.dct(logE, axis=0, type=2, norm='ortho')[:n_cep, :]
    # transpose to (T, D)
    return cep.T.astype(np.float32)

def run_lfcc_on_manifest(manifest_csv, out_dir: Path, pooled=True, save_frames=False, att_type = None):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(manifest_csv)
    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"LFCC {manifest_csv.name}"):
        utt = r['utt']
        wav = r['wav']
        if att_type:
            attack = r["attack_type"]
        label = int(r['label'])
        try:
            frames = extract_lfcc_from_wav(wav)
            if save_frames:
                np.save(out_dir / f"{utt}_frames.npy", frames)
            vec = pool_mean_std(frames) if pooled else frames
            np.save(out_dir / f"{utt}.npy", vec)
            if att_type:
                rows.append((utt, str((out_dir / f'{utt}.npy').resolve()), label, attack))
            else:
                rows.append((utt, str((out_dir / f'{utt}.npy').resolve()), label))
        except Exception as e:
            print(f"Failed {utt}: {e}")
    out_manifest = out_dir.parent / f"{out_dir.name}_{manifest_csv.stem}_feat_manifest.csv"
    if att_type:
        pd.DataFrame(rows, columns=["utt","feat_path","label", "attack_type"]).to_csv(out_manifest, index=False)
    else:
        pd.DataFrame(rows, columns=["utt","feat_path","label"]).to_csv(out_manifest, index=False)
    print("Wrote LFCC features to", out_dir, "and manifest", out_manifest)
    return out_manifest

In [ ]:
# lfcc_train_manifest = run_lfcc_on_manifest(TRAIN_PRE, FEATURES_DIR / "lfcc", pooled=True, save_frames=False)

In [ ]:
# lfcc_dev_manifest = run_lfcc_on_manifest(DEV_PRE, FEATURES_DIR / "lfcc", pooled=True, save_frames=False)

In [ ]:
# lfcc_eval_manifest = run_lfcc_on_manifest(EVAL_PRE, FEATURES_DIR / "lfcc", pooled=True, save_frames=False,att_type=True)

## CQCC

In [ ]:
def extract_cqcc_from_wav(wav_path, sr=SR, n_cq_bins=84, hop_length=HOP_LENGTH, n_cep=CQCC_N_CEP):
    """
    Simple CQCC-like extractor:
      - compute CQT magnitude (log)
      - treat log-CQT like spectral representation, apply DCT across freq bins
      - keep first n_cep coefficients per frame
    n_cq_bins: number of CQT bins (recommend ~84 or 96)
    """
    y, sr0 = librosa.load(wav_path, sr=sr, mono=True)
    # CQT: returns (freq_bins, frames)
    C = librosa.cqt(y, sr=sr, hop_length=hop_length, n_bins=n_cq_bins, bins_per_octave=12)
    C_mag = np.abs(C)
    # convert to power or magnitude, avoid zeros
    C_mag[C_mag <= 1e-12] = 1e-12
    logC = np.log(C_mag)
    # Now apply DCT across frequency axis (axis=0) to get cepstral-like coefficients
    # logC: (freq_bins, T)
    cep = fftpack.dct(logC, axis=0, type=2, norm='ortho')[:n_cep, :]
    return cep.T.astype(np.float32)

def run_cqcc_on_manifest(manifest_csv, out_dir: Path, pooled=True, save_frames=False, att_type = None):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(manifest_csv)
    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"CQCC {manifest_csv.name}"):
        utt = r['utt']
        wav = r['wav']
        if att_type:
            attack = r['attack_type']
        label = int(r['label'])
        try:
            frames = extract_cqcc_from_wav(wav)
            if save_frames:
                np.save(out_dir / f"{utt}_frames.npy", frames)
            vec = pool_mean_std(frames) if pooled else frames
            np.save(out_dir / f"{utt}.npy", vec)
            if att_type:
                rows.append((utt, str((out_dir / f'{utt}.npy').resolve()), label, attack))
            else:
                rows.append((utt, str((out_dir / f'{utt}.npy').resolve()), label))
        except Exception as e:
            print(f"Failed {utt}: {e}")
    out_manifest = out_dir.parent / f"{out_dir.name}_{manifest_csv.stem}_feat_manifest.csv"
    if att_type:
        pd.DataFrame(rows, columns=["utt","feat_path","label", "attack_type"]).to_csv(out_manifest, index=False)
    else:
        pd.DataFrame(rows, columns=["utt","feat_path","label"]).to_csv(out_manifest, index=False)
    print("Wrote CQCC-like features to", out_dir, "and manifest", out_manifest)
    return out_manifest

In [ ]:

# cqcc_train_manifest = run_cqcc_on_manifest(TRAIN_PRE, FEATURES_DIR / "cqcc", pooled=True, save_frames=False)

In [ ]:
# cqcc_dev_manifest = run_cqcc_on_manifest(DEV_PRE, FEATURES_DIR / "cqcc", pooled=True, save_frames=False)

In [ ]:
# cqcc_eval_manifest = run_cqcc_on_manifest(EVAL_PRE, FEATURES_DIR / "cqcc", pooled=True, save_frames=False,att_type=True)

## Deep Embeddings

Run these cells to extract the deep embeddings

In [ ]:
from pathlib import Path
from functools import partial

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
# from transformers import AutoProcessor, AutoModel
from transformers import AutoFeatureExtractor, AutoModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SR = 16000


def load_audio(path, sr=SR):
    import librosa
    y, _ = librosa.load(str(path), sr=sr, mono=True)
    return y.astype(np.float32)


def pool_mean(x):
    return x.mean(axis=0).astype(np.float32)


def pool_meanstd(x):
    return np.concatenate([x.mean(axis=0), x.std(axis=0)], axis=0).astype(np.float32)


# def load_ssl_model(model_name: str):
#     processor = AutoProcessor.from_pretrained(model_name)
#     model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()
#     return processor, model

def load_ssl_model(model_name: str):
    feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()
    return feature_extractor, model


# @torch.no_grad()
# def extract_ssl_embedding_from_wav(
#     wav_path,
#     processor,
#     model,
#     pooling: str = "meanstd",
#     sr: int = SR,
# ):
#     y = load_audio(wav_path, sr=sr)

#     inputs = processor(y, sampling_rate=sr, return_tensors="pt", padding=True)
#     inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

#     out = model(**inputs)
#     hidden = out.last_hidden_state.squeeze(0).detach().cpu().numpy()  # (T, D)

#     if pooling == "mean":
#         return pool_mean(hidden)
#     elif pooling == "meanstd":
#         return pool_meanstd(hidden)
#     else:
#         raise ValueError("pooling must be 'mean' or 'meanstd'")


@torch.no_grad()
def extract_ssl_embedding_from_wav(
    wav_path,
    feature_extractor,
    model,
    pooling: str = "meanstd",
    sr: int = SR,
):
    y = load_audio(wav_path, sr=sr)

    inputs = feature_extractor(
        y,
        sampling_rate=sr,
        return_tensors="pt",
        padding=True,
        return_attention_mask=True,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    out = model(**inputs)
    hidden = out.last_hidden_state.squeeze(0).detach().cpu().numpy()  # (T, D)

    if pooling == "mean":
        return pool_mean(hidden)
    elif pooling == "meanstd":
        return pool_meanstd(hidden)
    else:
        raise ValueError("pooling must be 'mean' or 'meanstd'")

def extract_and_save_ssl_embeddings(
    manifest_csv: str | Path,
    out_dir: str | Path,
    model_name: str,
    pooling: str = "meanstd",
    att_type = None
):
    """
    Input manifest: utt,wav,label
    Output manifest: utt,feat_path,label
    """
    manifest_csv = Path(manifest_csv)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    feature_extractor, model = load_ssl_model(model_name)
    fn = partial(
        extract_ssl_embedding_from_wav,
        # processor=processor,
        feature_extractor=feature_extractor,
        model=model,
        pooling=pooling,
        sr=SR,
    )

    df = pd.read_csv(manifest_csv)
    required = {"utt", "wav", "label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Audio manifest missing columns: {missing}")

    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"Extracting {out_dir.name}"):
        utt = str(r["utt"])
        wav = r["wav"]
        label = int(r["label"])
        if att_type:
            attack = r["attack_type"]

        try:
            vec = fn(wav)
            feat_path = out_dir / f"{utt}.npy"
            np.save(feat_path, vec.astype(np.float32))
            if att_type:
                rows.append((utt, str(feat_path.resolve()), label, attack))
            else:
                rows.append((utt, str(feat_path.resolve()), label))
        except Exception as e:
            print(f"[FAIL] {utt}: {e}")

    out_manifest = out_dir.parent / f"{out_dir.name}_{manifest_csv.stem}_feat_manifest.csv"
    if att_type:
        pd.DataFrame(rows, columns=["utt", "feat_path", "label", "attack_type"]).to_csv(out_manifest, index=False)
    else:
        pd.DataFrame(rows, columns=["utt", "feat_path", "label"]).to_csv(out_manifest, index=False)
    return out_manifest

In [ ]:
# # Wav2Vec Training
# wav2vec_dir = Path("/scratch1/kodachi/p2/features/wav2vec2")
# wav2vec_train = extract_and_save_ssl_embeddings(
#     TRAIN_PRE,
#     wav2vec_dir,
#     "facebook/wav2vec2-base",
#     pooling="meanstd",
# )

In [ ]:
# # Wav2Vec Dev
# wav2vec_dev = extract_and_save_ssl_embeddings(
#     DEV_PRE,
#     wav2vec_dir,
#     "facebook/wav2vec2-base",
#     pooling="meanstd",
# )

In [ ]:
# # Wav2Vec Eval
# wav2vec_eval = extract_and_save_ssl_embeddings(
#     EVAL_PRE,
#     wav2vec_dir,
#     "facebook/wav2vec2-base",
#     pooling="meanstd",
#     att_type=True
# )

In [ ]:
# # Hubert Training
# hubert_dir = Path("/scratch1/kodachi/p2/features/hubert")
# hubert_train = extract_and_save_ssl_embeddings(
#     TRAIN_PRE,
#     hubert_dir,
#     "facebook/hubert-base-ls960",
#     pooling="meanstd",
# )

In [ ]:
# # Hubert Dev
# hubert_dev = extract_and_save_ssl_embeddings(
#     DEV_PRE,
#     hubert_dir,
#     "facebook/hubert-base-ls960",
#     pooling="meanstd",
# )

In [ ]:
# # Hubert Eval
# hubert_eval = extract_and_save_ssl_embeddings(
#     EVAL_PRE,
#     hubert_dir,
#     "facebook/hubert-base-ls960",
#     pooling="meanstd",
#     att_type=True
# )

## Wavelet Scattering Transform

In [ ]:
import librosa
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from scipy import fftpack
from kymatio.numpy import Scattering1D

import scipy, kymatio
print(scipy.__version__)
print(kymatio.__version__)

# -----------------------------
# Config
# -----------------------------
SR = 16000
TARGET_SEC = 4.0
TARGET_SAMPLES = int(SR * TARGET_SEC)

# Scattering parameters
SCATTER_J = 6
SCATTER_Q = 8
SCATTER_MAX_ORDER = 2

# -----------------------------
# Helpers
# -----------------------------
def pool_mean_std(x, eps=1e-8):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 1:
        return x
    mu = x.mean(axis=0)
    sd = x.std(axis=0)
    return np.concatenate([mu, sd], axis=0).astype(np.float32)

def load_audio_fixed_len(wav_path, sr=SR, target_samples=TARGET_SAMPLES):
    y, _ = librosa.load(wav_path, sr=sr, mono=True)
    y = y.astype(np.float32)

    if len(y) < target_samples:
        y = librosa.util.fix_length(y, size=target_samples)
    else:
        y = y[:target_samples]

    return y

# -----------------------------
# Scattering extractor
# -----------------------------
def extract_scattering_from_wav(
    wav_path,
    scattering=None,
    sr=SR,
    target_samples=TARGET_SAMPLES,
):
    """
    Returns frames of shape (T, D) so it matches your MFCC-style pipeline.
    """
    y = load_audio_fixed_len(wav_path, sr=sr, target_samples=target_samples)

    # Create once and reuse outside the loop if you want speed.
    if scattering is None:
        scattering = Scattering1D(
            J=SCATTER_J,
            shape=target_samples,
            Q=SCATTER_Q,
            max_order=SCATTER_MAX_ORDER,
            out_type="array",
        )

    # Kymatio returns array shaped (C, T); transpose to (T, C)
    Sx = scattering(y)
    if hasattr(Sx, "detach"):
        Sx = Sx.detach().cpu().numpy()

    return Sx.T.astype(np.float32)

# -----------------------------
# Manifest runner
# -----------------------------
def run_scattering_on_manifest(
    manifest_csv,
    out_dir: Path,
    pooled=True,
    save_frames=False,
    att_type=None,
):
    """
    Input manifest:
      - if att_type is False: utt,wav,label
      - if att_type is True:  utt,wav,label,attack_type

    Output manifest:
      - utt,feat_path,label
      - or utt,feat_path,label,attack_type
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(manifest_csv)

    required = {"utt", "wav", "label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Manifest missing columns: {missing}")

    if att_type and "attack_type" not in df.columns:
        raise ValueError("att_type=True, but manifest does not contain attack_type")

    # Create once and reuse for all files
    scattering = Scattering1D(
        J=SCATTER_J,
        shape=TARGET_SAMPLES,
        Q=SCATTER_Q,
        max_order=SCATTER_MAX_ORDER,
        out_type="array",
    )

    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"Scattering {Path(manifest_csv).name}"):
        utt = r["utt"]
        wav = r["wav"]
        label = int(r["label"])
        attack = r["attack_type"] if att_type else None

        try:
            frames = extract_scattering_from_wav(
                wav_path=wav,
                scattering=scattering,
                sr=SR,
                target_samples=TARGET_SAMPLES,
            )

            if save_frames:
                np.save(out_dir / f"{utt}_frames.npy", frames)

            vec = pool_mean_std(frames) if pooled else frames
            feat_path = out_dir / f"{utt}.npy"
            np.save(feat_path, vec)

            if att_type:
                rows.append((utt, str(feat_path.resolve()), label, attack))
            else:
                rows.append((utt, str(feat_path.resolve()), label))

        except Exception as e:
            print(f"Failed {utt}: {e}")

    out_manifest = out_dir.parent / f"{out_dir.name}_{Path(manifest_csv).stem}_feat_manifest.csv"

    if att_type:
        pd.DataFrame(rows, columns=["utt", "feat_path", "label", "attack_type"]).to_csv(out_manifest, index=False)
    else:
        pd.DataFrame(rows, columns=["utt", "feat_path", "label"]).to_csv(out_manifest, index=False)

    print("Wrote scattering features to", out_dir, "and manifest", out_manifest)
    return out_manifest

In [ ]:
scatter_dir = Path("/scratch1/kodachi/p2/features/scatter")
# train_scatter_manifest = run_scattering_on_manifest(
#     TRAIN_PRE,
#     scatter_dir,
#     pooled=True,
#     save_frames=False,
#     att_type=False,
# )

In [ ]:
dev_scatter_manifest = run_scattering_on_manifest(
    DEV_PRE,
    scatter_dir,
    pooled=True,
    save_frames=False,
    att_type=False,
)

In [ ]:
eval_scatter_manifest = run_scattering_on_manifest(
    EVAL_PRE,
    scatter_dir,
    pooled=True,
    save_frames=False,
    att_type=True,
)

## Add attack types to Eval Manifests

In [ ]:
# import pandas as pd

# # Load the CSVs
# df_main = pd.read_csv("manifests/lfcc_LA_eval_preprocessed_feat_manifest.csv")   # the one you want to ADD the column to
# df_attack = pd.read_csv("manifests/LA_eval_preprocessed.csv") # the one that HAS 'attack_type'

# # Merge on 'utt'
# df_merged = df_main.merge(
#     df_attack[['utt', 'attack_type']], 
#     on='utt', 
#     how='left'  # keeps all rows from df_main
# )

# # Save to new CSV
# f = "manifests/lfcc_LA_eval_final.csv"
# df_merged.to_csv(f, index=False)

# print(f"Done! Saved as {f}")